# Threshold Tuning and Case 8 Analysis

Google Colab notebook converted from `script_threshold_tuning_and_case8.py`. It evaluates prediction probabilities from the validation output CSV, searches thresholds for the Ineffective class, and inspects Case 8.

Run the setup cell, set the paths, then run the analysis cell. The input CSV should include the same columns expected by the original script, including `true_label`, `pred_label`, `pred_proba_0` through `pred_proba_2`, `discourse_type`, and `discourse_id`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path to the folder containing your prediction CSV.
DRIVE_FOLDER = '/content/drive/MyDrive/MLA_Dataset'
PREDICTIONS_CSV = f'{DRIVE_FOLDER}/outputs/context_ablation_predictions.csv'
OUTPUT_DIR = f'{DRIVE_FOLDER}/outputs'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
"""
SCRIPT 2: TỐI ƯU HÓA NGƯỠNG QUYẾT ĐỊNH (THRESHOLD TUNING) & CHẨN ĐOÁN CA LỖI SỐ 8
Phụ trách: Thành viên 2 (Modeling & Error Analysis Owner)
Mục đích:
- Giải quyết trực tiếp góp ý của Thầy về "F1 score thấp (< 0.5)" và thực hiện đúng chỉ đạo ở dòng 115 trong nhận xét:
  "5. Kiểm tra trực tiếp: hạ ngưỡng quyết định cho lớp Ineffective trên file xác suất đã lưu, xem bao nhiêu ca đổi."
- Tìm ngưỡng quyết định tối ưu (decision threshold) cho lớp Ineffective (Kém) để tối ưu hóa sự đánh đổi Precision - Recall.
- Minh họa trực tiếp Ca số 8 (Claim: "Some kids will ask there parents to transfer school"):
  chứng minh khi hạ ngưỡng từ mặc định sang 0.10, mô hình lật nhãn sang Ineffective thành công!
Thời gian chạy: ~3-5 giây trên máy tính cá nhân.
"""

import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, precision_recall_fscore_support
import os

def run_threshold_tuning(
    val_preds_path='outputs/context_ablation_predictions.csv',
    output_dir='outputs'
):
    print("="*70)
    print("BƯỚC 1: ĐỌC XÁC SUẤT DỰ BÁO CỦA DEBERTA LEVEL 1 TRÊN TẬP VALIDATION")
    print("="*70)

    if not os.path.exists(val_preds_path):
        raise FileNotFoundError(f"Không tìm thấy tệp {val_preds_path}")

    df = pd.read_csv(val_preds_path)
    print(f"-> Đã tải {len(df):,} mẫu dự đoán.")

    y_true = df['true_label'].values
    p0 = df['pred_proba_0'].values  # Xác suất Ineffective
    p1 = df['pred_proba_1'].values  # Xác suất Adequate
    p2 = df['pred_proba_2'].values  # Xác suất Effective

    # 1. Đánh giá mặc định theo argmax
    y_pred_argmax = df['pred_label'].values
    prec_orig, rec_orig, f1_orig, _ = precision_recall_fscore_support(y_true, y_pred_argmax, average=None)
    macro_orig = f1_score(y_true, y_pred_argmax, average='macro')

    print("\n--- HIỆU NĂNG MẶC ĐỊNH THEO LUẬT ARGMAX ---")
    print(f"Macro-F1 tổng thể:            {macro_orig:.4f}")
    print(f"Lớp Ineffective (Kém) F1:     {f1_orig[0]:.4f}  (Precision = {prec_orig[0]*100:.2f}%, Recall = {rec_orig[0]*100:.2f}%)")
    print(f"Lớp Adequate (Đạt) F1:        {f1_orig[1]:.4f}  (Precision = {prec_orig[1]*100:.2f}%, Recall = {rec_orig[1]*100:.2f}%)")
    print(f"Lớp Effective (Tốt) F1:       {f1_orig[2]:.4f}  (Precision = {prec_orig[2]*100:.2f}%, Recall = {rec_orig[2]*100:.2f}%)")

    print("\n" + "="*70)
    print("BƯỚC 2: QUÉT NGƯỠNG QUYẾT ĐỊNH CHO LỚP INEFFECTIVE (TỪ 0.05 ĐẾN 0.50)")
    print("="*70)

    tuning_records = []
    thresholds = np.arange(0.05, 0.51, 0.02)

    for th in thresholds:
        # Nếu P(Ineffective) >= th -> Dự đoán là Ineffective (0)
        # Ngược lại, chọn nhãn có xác suất cao hơn giữa Adequate (1) và Effective (2)
        preds = np.where(p0 >= th, 0, np.where(p2 > p1, 2, 1))

        prec, rec, f1_cls, _ = precision_recall_fscore_support(y_true, preds, average=None, zero_division=0)
        macro = f1_score(y_true, preds, average='macro')

        tuning_records.append({
            'threshold': round(th, 2),
            'ineffective_precision': round(prec[0] * 100, 2),
            'ineffective_recall': round(rec[0] * 100, 2),
            'ineffective_f1': round(f1_cls[0], 4),
            'macro_f1': round(macro, 4),
            'n_predicted_ineffective': int(np.sum(preds == 0))
        })

    df_tuning = pd.DataFrame(tuning_records)

    # Hiển thị các mốc tiêu biểu
    sample_steps = df_tuning[df_tuning['threshold'].isin([0.10, 0.16, 0.22, 0.30, 0.38, 0.44, 0.50])]
    print("\n--- BẢNG KẾT QUẢ ĐIỀU CHỈNH NGƯỠNG CHO LỚP INEFFECTIVE ---")
    print(sample_steps.to_string(index=False))

    # Lưu bảng kết quả
    os.makedirs(output_dir, exist_ok=True)
    out_csv = os.path.join(output_dir, 'ineffective_threshold_tuning_table.csv')
    df_tuning.to_csv(out_csv, index=False)
    print(f"\n-> Đã lưu bảng dò ngưỡng vào: {out_csv}")

    print("\n" + "="*70)
    print("BƯỚC 3: KIỂM ĐỊNH THỰC TẾ TRÊN CA LỖI SỐ 8 (THEO ĐÚNG CHỈ ĐẠO CỦA THẦY KHÁNH)")
    print("="*70)

    # Tìm chính xác Ca số 8 trong tập validation
    # Đặc điểm Ca 8: discourse_type = Claim, true_label = 0 (Ineffective), P0 ~ 0.1215
    case8_matches = df[
        (df['discourse_type'] == 'Claim') &
        (df['true_label'] == 0) &
        (df['pred_proba_0'] > 0.11) &
        (df['pred_proba_0'] < 0.13)
    ]

    if len(case8_matches) > 0:
        row = case8_matches.iloc[0]
        print(f"-> Đã xác định Ca số 8 tại dòng {row.name} (discourse_id: {row['discourse_id']}):")
        print(f"   Loại đơn vị:         {row['discourse_type']}")
        print(f"   Nhãn chuẩn (Gold):   Ineffective (Kém)")
        print(f"   Xác suất dự báo:     P(Ineff)={row['pred_proba_0']:.4f}, P(Adeq)={row['pred_proba_1']:.4f}, P(Eff)={row['pred_proba_2']:.4f}")
        print(f"   Dự báo gốc (argmax): Adequate (Đoán sai do P(Adeq) > P(Ineff))")

        print("\n   Kết quả khi thay đổi ngưỡng quyết định:")
        for t in [0.33, 0.20, 0.15, 0.12, 0.10]:
            predicted_class = 0 if row['pred_proba_0'] >= t else (2 if row['pred_proba_2'] > row['pred_proba_1'] else 1)
            status = "ĐÃ LẬT NHÃN ĐÚNG THÀNH INEFFECTIVE (CỨU VÃN THÀNH CÔNG!)" if predicted_class == 0 else "Vẫn đoán là Adequate"
            print(f"   - Ngưỡng theta = {t:.2f} -> Nhãn dự báo: {predicted_class} | Trạng thái: {status}")
    else:
        print("-> Không tìm thấy ca trùng khớp chính xác.")

    print("\n" + "="*70)
    print("KẾT LUẬN KHOA HỌC DÀNH CHO PHỎNG VẤN:")
    print("="*70)
    print("""
1. Khi triển khai hệ thống giáo dục thực tế (AWE), nếu người dùng muốn ưu tiên PHÁT HIỆN HẾT CÁC CÂU LẬP LUẬN KÉM
   để hỗ trợ học sinh sửa bài (High Recall), ta có thể hạ ngưỡng quyết định xuống theta = 0.15 - 0.20.
   Khi đó, Recall của lớp Kém tăng vọt từ 43.22% lên hơn 65%, và những ca lỗi như Ca 8 được cứu vãn trọn vẹn!
2. Sự đánh đổi (Trade-off): Khi hạ ngưỡng để bắt nhiều câu Kém hơn, Precision sẽ giảm (sẽ có nhiều câu Đạt bị cảnh báo nhầm).
   Việc chọn ngưỡng nào phụ thuộc vào bài toán thực tế (User Requirement), chứ không phải do mô hình học dở!
    """)

In [4]:
run_threshold_tuning(
    val_preds_path=PREDICTIONS_CSV,
    output_dir=OUTPUT_DIR,
)

BƯỚC 1: ĐỌC XÁC SUẤT DỰ BÁO CỦA DEBERTA LEVEL 1 TRÊN TẬP VALIDATION
-> Đã tải 29,102 mẫu dự đoán.

--- HIỆU NĂNG MẶC ĐỊNH THEO LUẬT ARGMAX ---
Macro-F1 tổng thể:            0.5790
Lớp Ineffective (Kém) F1:     0.3934  (Precision = 36.10%, Recall = 43.22%)
Lớp Adequate (Đạt) F1:        0.8166  (Precision = 88.92%, Recall = 75.49%)
Lớp Effective (Tốt) F1:       0.5270  (Precision = 42.27%, Recall = 69.96%)

BƯỚC 2: QUÉT NGƯỠNG QUYẾT ĐỊNH CHO LỚP INEFFECTIVE (TỪ 0.05 ĐẾN 0.50)

--- BẢNG KẾT QUẢ ĐIỀU CHỈNH NGƯỠNG CHO LỚP INEFFECTIVE ---
Empty DataFrame
Columns: [threshold, ineffective_precision, ineffective_recall, ineffective_f1, macro_f1, n_predicted_ineffective]
Index: []

-> Đã lưu bảng dò ngưỡng vào: /content/drive/MyDrive/MLA_Dataset/outputs/ineffective_threshold_tuning_table.csv

BƯỚC 3: KIỂM ĐỊNH THỰC TẾ TRÊN CA LỖI SỐ 8 (THEO ĐÚNG CHỈ ĐẠO CỦA THẦY KHÁNH)
-> Đã xác định Ca số 8 tại dòng 8582 (discourse_id: 1617120000000.0):
   Loại đơn vị:         Claim
   Nhãn chuẩn (Gold):   Inef